In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

df = pd.read_csv('reasoning_summary_per_episode.csv')
# in each cell is a list of thinking event, extracted from `interactions_with_thinking.json`, first turn only
# {
#     "from": "Player 1",
#     "to": "Player 1",
#     "timestamp": "2025-09-07T00:12:18.120277",
#     "action": {
#         "type": "thinking",
#         "content": "..."
#     }
# },
# df["reasoning_raw"] = df["reasoning_raw"].apply(json.loads)
# list of thinking event -> list of list of labels
# df["reasoning_labels"] = df["reasoning_labels"].apply(json.loads)
df.head(10)

,lang,model,game,experiment,episode,reasoning_raw,reasoning_labels,reasoning_chain_len,reasoning_chain_avg_len,reasoning_chain_ratio,n_segment,cycle_edge_ratio,n_cycle_edges,n_total_edges
0,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00005,"[{'from': 'Player 1', 'to': 'Player 1', 'times...",[['ASSERT']],[1],1.000000,0.026316,0.000000,0.000000,0.000000,0.000000
1,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00002,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT', 'UNDERMINE', 'PROPOSE', 'PROPOSE']]",[4],4.000000,0.129032,1.000000,0.500000,1.000000,2.000000
2,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00003,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT', 'ASSERT']]",[2],2.000000,0.040000,0.000000,0.000000,0.000000,0.000000
3,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00004,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT'], ['PROPOSE', 'PROPOSE', 'PROPOSE',...","[1, 4, 2]",2.333333,0.207246,0.333333,0.333333,0.666667,0.666667
4,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00001,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT', 'PROPOSE']]",[2],2.000000,0.064516,1.000000,0.000000,0.000000,0.000000
5,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00000,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT'], ['PROPOSE', 'PROPOSE', 'PROPOSE',...","[1, 5, 3]",3.000000,0.215697,0.666667,0.666667,1.666667,1.666667
6,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00005,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT', 'ASSERT'], ['PROPOSE', 'PROPOSE', ...","[2, 4]",3.000000,0.228571,0.500000,0.500000,1.500000,1.500000
7,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00002,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['ASSERT', 'ASSERT', 'PROPOSE'], ['PROPOSE', ...","[3, 5]",4.000000,0.321895,1.000000,0.500000,2.000000,2.000000
8,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00003,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['UNDERMINE', 'PROPOSE']]",[2],2.000000,0.054054,1.000000,0.000000,0.000000,1.000000
9,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00004,"[{'from': 'Player 1', 'to': 'Player 1', 'times...","[['CONCLUDE', 'ASSERT', 'PROPOSE'], ['PROPOSE'...","[3, 3]",3.000000,0.162260,1.000000,0.500000,1.000000,1.000000


In [2]:
# drop "reasoning_raw" and "reasoning_labels"
df = df.drop(columns=["reasoning_raw", "reasoning_labels"])
df.head(10)

,lang,model,game,experiment,episode,reasoning_chain_len,reasoning_chain_avg_len,reasoning_chain_ratio,n_segment,cycle_edge_ratio,n_cycle_edges,n_total_edges
0,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00005,[1],1.000000,0.026316,0.000000,0.000000,0.000000,0.000000
1,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00002,[4],4.000000,0.129032,1.000000,0.500000,1.000000,2.000000
2,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00003,[2],2.000000,0.040000,0.000000,0.000000,0.000000,0.000000
3,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00004,"[1, 4, 2]",2.333333,0.207246,0.333333,0.333333,0.666667,0.666667
4,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00001,[2],2.000000,0.064516,1.000000,0.000000,0.000000,0.000000
5,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_negotiation_hard,instance_00000,"[1, 5, 3]",3.000000,0.215697,0.666667,0.666667,1.666667,1.666667
6,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00005,"[2, 4]",3.000000,0.228571,0.500000,0.500000,1.500000,1.500000
7,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00002,"[3, 5]",4.000000,0.321895,1.000000,0.500000,2.000000,2.000000
8,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00003,[2],2.000000,0.054054,1.000000,0.000000,0.000000,1.000000
9,en,claude-sonnet-4-20250514-t0.0,hot_air_balloon,air_balloon_survival_en_reasoning off_hard,instance_00004,"[3, 3]",3.000000,0.162260,1.000000,0.500000,1.000000,1.000000
